# Day 044 — Exercise 1: setup_engine

**What you'll build:** `setup_engine(url='sqlite:///:memory:') -> Engine` — create a SQLAlchemy engine, configure it for in-memory SQLite, and call `Base.metadata.create_all(engine)` to create all ORM tables.

**Why it matters:** The engine is SQLAlchemy's entry point to the database. It holds the connection pool and knows which database to talk to via a URL. `create_all` inspects every class that inherits from `Base` and creates the corresponding tables — replacing the manual `CREATE TABLE IF NOT EXISTS` DDL from Day 42 with a single call.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'

## Your Implementation

In [ ]:
def setup_engine(url='sqlite:///:memory:'):
    """
    Create a SQLAlchemy engine and create all ORM-mapped tables.

    Use StaticPool and check_same_thread=False so that the in-memory
    SQLite database is shared across all connections from this engine.

    Returns:
        Engine — the connected engine with tables created
    """
    # TODO: engine = create_engine(
    # TODO:     url,
    # TODO:     connect_args={'check_same_thread': False},
    # TODO:     poolclass=StaticPool,
    # TODO: )
    # TODO: Base.metadata.create_all(engine)
    # TODO: return engine
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: function defined
    try:
        assert 'setup_engine' in globals()
        passed += 1; print('\u2705 Check 1: setup_engine is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns an Engine
    try:
        from sqlalchemy.engine import Engine
        engine = setup_engine()
        assert isinstance(engine, Engine), \
            f'expected Engine, got {type(engine).__name__}'
        passed += 1; print('\u2705 Check 2: returns an Engine')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: items table was created
    try:
        from sqlalchemy import inspect as sa_inspect
        inspector = sa_inspect(engine)
        tables = inspector.get_table_names()
        assert 'items' in tables, \
            f'items table not found; got {tables}'
        passed += 1; print('\u2705 Check 3: items table created')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: can open a Session
    try:
        with Session(engine) as session:
            assert session is not None
        passed += 1; print('\u2705 Check 4: Session opens without error')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: items table has correct columns
    try:
        from sqlalchemy import inspect as sa_inspect
        cols = {c['name'] for c in sa_inspect(engine).get_columns('items')}
        required = {'id', 'name', 'category', 'price', 'quantity'}
        assert required <= cols, \
            f'missing columns: {required - cols}'
        passed += 1; print(f'\u2705 Check 5: items has columns {required}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine
```

</details>